# Phase 6 — Retention, LTV, and Budget Allocation

**[SYNTHETIC DATA — Tablr portfolio simulation]**

**Period:** 2026-01-05 to 2026-03-29 | **Maturity reference:** 2026-09-13

This notebook answers the core Phase 6 question: **which acquisition sources produce users who reach value, return, and generate enough economic value to justify continued investment?**

## Sections
1. Setup & data loading
2. Cohort activation funnel (weekly cohort table)
3. M1/M3/M6 retention by channel (with maturity flags)
4. Observed LTV by channel and tier
5. LTV:CAC and payback analysis
6. Budget allocation scenario ($150K/month, 10% exploration)
7. Key findings: what changes when you add retention to the acquisition picture?

---
## 1. Setup & Data Loading

All analytics functions are imported from the `growth_agent.analytics` package. Raw data lives in `data/synthetic/*.parquet`. DuckDB is used for SQL aggregations; Pandas for display.

In [ ]:
import sys
import os
from pathlib import Path

# Add project src to Python path
project_root = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(project_root / 'src'))

import duckdb
import pandas as pd

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

DATA_DIR = project_root / 'data' / 'synthetic'
print('Project root:', project_root)
print('Data directory:', DATA_DIR)
print('Parquet files:', [f.name for f in DATA_DIR.glob('*.parquet')])

In [ ]:
from growth_agent.analytics.cohorts import build_cohorts, get_cohort_summary
from growth_agent.analytics.retention import get_retention_by_channel, get_retention_detail
from growth_agent.analytics.ltv import get_ltv_by_channel, get_ltv_estimates
from growth_agent.analytics.budget_allocator import (
    AllocationScenario, ChannelConstraint, allocate_budget
)

print('All analytics modules loaded successfully.')

---
## 2. Cohort Activation Funnel

**Cohort definition:** ISO week of TRIAL_SIGNUP event (W02–W13 2026, 12 weeks).

**Activation metric:** CAMPAIGN_LAUNCHED within 14 days of TRIAL_SIGNUP. This is the primary product activation milestone — the moment a restaurant owner actually uses Tablr to run a campaign.

**Right-censoring note:** Cohorts are flagged `is_m6_mature` if they have been observable for >= 180 days from the maturity reference date (2026-09-13). W02 (Jan 5) = 252 days ✓. W13 (Mar 23) = 174 days ✗.

In [ ]:
# Overall funnel summary
summary = get_cohort_summary(DATA_DIR)
print('=== Cohort Funnel Summary [SYNTHETIC DATA] ===')
for k, v in summary.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.1%}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# Weekly cohort table — aggregate across channel/persona for readability
cohort_rows = build_cohorts(DATA_DIR)

from collections import defaultdict

weekly = defaultdict(lambda: {
    'cohort_start_date': None, 'trials': 0, 'onboarding': 0,
    'activated_14d': 0, 'subscribed': 0, 'days_observed': 0, 'is_m6_mature': False
})

for r in cohort_rows:
    d = weekly[r.cohort_week]
    d['cohort_start_date'] = r.cohort_start_date
    d['trials'] += r.trials
    d['onboarding'] += r.onboarding_completed
    d['activated_14d'] += r.campaign_launched_14d
    d['subscribed'] += r.subscription_started
    d['days_observed'] = r.days_observed
    d['is_m6_mature'] = r.is_m6_mature

rows_weekly = []
for week, d in sorted(weekly.items()):
    trials = d['trials']
    activated = d['activated_14d']
    subscribed = d['subscribed']
    rows_weekly.append({
        'Cohort Week': week,
        'Start Date': str(d['cohort_start_date']),
        'Trials': trials,
        'Onboarding': d['onboarding'],
        'Activated (14d)': activated,
        'Subscribed': subscribed,
        'Activation Rate': f"{activated/trials:.1%}" if trials else 'n/a',
        'Conv Rate': f"{subscribed/trials:.1%}" if trials else 'n/a',
        'Days Observed': d['days_observed'],
        'M6 Mature': '✓' if d['is_m6_mature'] else '✗'
    })

df_weekly = pd.DataFrame(rows_weekly)
print('[SYNTHETIC DATA] Weekly Cohort Activation Funnel')
df_weekly

**Reading the table:**
- Activation rates are stable (23–28%) across all 12 cohort weeks — no activation decay
- Weeks marked ✗ for M6 Mature are right-censored: we cannot yet report their 6-month retention
- Conversion rates (subscription / trials) average ~12.9% overall

---
## 3. M1/M3/M6 Retention by Channel

**Retention logic (subscription-level, not trial-level):**
- **M1:** `status = 'active'` OR `churn_date > conversion_date + 30 days`
- **M3:** Same logic, 90-day window
- **M6:** Same logic, 180-day window — **only counted when `conversion_date + 180 ≤ 2026-09-13`**

**Right-censoring note:** All channels show `m6_immature_warning = True` because April–May 2026 conversions are not yet observable for 180 days. M6 rates are computed on the eligible subset only.

In [ ]:
retention_by_channel = get_retention_by_channel(DATA_DIR)

ret_rows = []
for r in retention_by_channel:
    ret_rows.append({
        'Channel': r.channel,
        'Subscribers': r.subscribers,
        'M1 Rate': f"{r.m1_rate:.1%}" if r.m1_rate else 'n/a',
        'M3 Rate': f"{r.m3_rate:.1%}" if r.m3_rate else 'n/a',
        'M6 Rate (mature)': f"{r.m6_rate:.1%}" if r.m6_rate else 'n/a',
        'M6 Eligible Subs': r.m6_mature_subscribers,
        'M6 Immature Warning': '⚠️' if r.m6_immature_warning else ''
    })

df_ret = pd.DataFrame(ret_rows)
print('[SYNTHETIC DATA] Subscription Retention by Channel')
df_ret

In [ ]:
# Monthly cohort detail — M6 maturity per cohort-month
detail_rows = get_retention_detail(DATA_DIR)

# Summarize at cohort_month level (collapse channel/persona)
from collections import defaultdict
month_agg = defaultdict(lambda: {'subs': 0, 'm1': 0, 'm3': 0, 'm6': 0, 'm6_elig': 0, 'immature': False})
for row in detail_rows:
    m = row.cohort_month
    month_agg[m]['subs'] += row.subscribers
    month_agg[m]['m1'] += row.m1_retained
    month_agg[m]['m3'] += row.m3_retained
    month_agg[m]['m6'] += row.m6_retained
    month_agg[m]['m6_elig'] += 0  # track from subscriptions count logic
    if not row.m6_is_mature:
        month_agg[m]['immature'] = True

mo_rows = []
for month, d in sorted(month_agg.items()):
    subs = d['subs']
    mo_rows.append({
        'Cohort Month': month,
        'Subscribers': subs,
        'M1 Rate': f"{d['m1']/subs:.1%}" if subs else 'n/a',
        'M3 Rate': f"{d['m3']/subs:.1%}" if subs else 'n/a',
        'M6 Status': '⚠️ Immature' if d['immature'] else '✓ Mature'
    })

df_monthly = pd.DataFrame(mo_rows)
print('[SYNTHETIC DATA] Monthly Cohort Retention Summary')
df_monthly

**Key observation:** TIKTOK leads on all three retention windows (M1: 85.3%, M3: 60.7%, M6: 47.4%). UNKNOWN channel (missing attribution) shows the weakest downstream retention — M3 at 47.2% vs 55–61% for tracked channels. This is consistent with organic/direct traffic being lower-intent than paid.

---
## 4. Observed LTV by Channel and Tier

**Observed LTV** = sum of `fact_revenue_events.amount_usd` per subscriber account, averaged by segment.

**Important caveat:** The simulation covers only 2026-01-05 to 2026-03-29. Revenue events exist only at `month_num` 0, 1, and 2. Observed LTV is therefore significantly understated relative to true steady-state LTV.

In [ ]:
ltv_ests = get_ltv_estimates(DATA_DIR)

# Channel-level summary (collapse persona segments)
channel_ltv = defaultdict(lambda: {'subs': 0, 'price_sum': 0.0, 'obs_sum': 0.0, 'proj_sum': 0.0, 'proj_cnt': 0})
for e in ltv_ests:
    d = channel_ltv[e.channel]
    d['subs'] += e.subscribers
    d['price_sum'] += e.avg_monthly_price_usd * e.subscribers
    d['obs_sum'] += e.observed_avg_ltv_usd * e.subscribers
    if e.projected_ltv_usd:
        d['proj_sum'] += e.projected_ltv_usd * e.subscribers
        d['proj_cnt'] += e.subscribers

ltv_rows_ch = []
for ch, d in sorted(channel_ltv.items()):
    subs = d['subs']
    avg_price = d['price_sum'] / subs if subs else 0
    avg_obs = d['obs_sum'] / subs if subs else 0
    avg_proj = d['proj_sum'] / d['proj_cnt'] if d['proj_cnt'] > 0 else None
    ltv_rows_ch.append({
        'Channel': ch,
        'Subscribers': subs,
        'Avg Monthly Price': f'${avg_price:.0f}',
        'Observed Avg LTV': f'${avg_obs:.2f}',
        'Projected LTV (const churn)': f'${avg_proj:.2f}' if avg_proj else 'n/a',
        'Note': '2-3 months observed only'
    })

df_ltv_ch = pd.DataFrame(ltv_rows_ch)
print('[SYNTHETIC DATA] Observed and Projected LTV by Channel')
df_ltv_ch

In [ ]:
# LTV by subscription tier (collapse channel/persona)
tier_ltv = defaultdict(lambda: {'subs': 0, 'price_sum': 0.0, 'obs_sum': 0.0, 'churn_sum': 0.0})
for e in ltv_ests:
    # We can't easily get tier from estimates (estimates are by channel/persona)
    # Query directly from DuckDB instead
    pass

# Direct DuckDB query for tier-level LTV
con = duckdb.connect()
subs_path = str(DATA_DIR / 'fact_subscriptions.parquet')
rev_path = str(DATA_DIR / 'fact_revenue_events.parquet')

df_tier = con.execute(f"""
    WITH rev AS (
        SELECT account_id, SUM(amount_usd) AS total_rev
        FROM read_parquet('{rev_path}')
        GROUP BY account_id
    )
    SELECT
        s.subscription_tier,
        COUNT(*) AS subscribers,
        AVG(s.monthly_price_usd) AS avg_monthly_price,
        AVG(COALESCE(r.total_rev, 0)) AS observed_avg_ltv,
        SUM(CASE WHEN s.status = 'churned' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS churn_rate,
        AVG(s.monthly_price_usd) / NULLIF(
            SUM(CASE WHEN s.status = 'churned' THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 0
        ) AS projected_ltv
    FROM read_parquet('{subs_path}') s
    LEFT JOIN rev r ON s.account_id = r.account_id
    GROUP BY s.subscription_tier
    ORDER BY avg_monthly_price DESC
""").df()

con.close()
print('[SYNTHETIC DATA] LTV by Subscription Tier')
print('Note: Observed LTV covers ~2-3 months of revenue only (short simulation)')
df_tier

---
## 5. LTV:CAC and Payback Analysis

**CPAO (Cost Per Activated Owner):** Total paid spend ÷ count of accounts that launched a campaign within 14 days of trial signup. This is the preferred CAC denominator because it measures users who reached product value, not just trial signups.

**LTV:CAC:** Observed LTV ÷ CPAO. Values below 1.0x reflect the short revenue observation window, not an uneconomic business.

**Payback period:** CPAO ÷ Monthly price (months to recover customer acquisition cost).

In [ ]:
ltv_by_ch = get_ltv_by_channel(DATA_DIR)

ltv_cac_rows = []
for l in sorted(ltv_by_ch, key=lambda x: (x.cpao_usd or 999999)):
    ltv_cac_rows.append({
        'Channel': l.channel,
        'CPAO': f'${l.cpao_usd:.0f}' if l.cpao_usd else 'n/a',
        'Avg Monthly Price': f'${l.avg_monthly_price_usd:.0f}',
        'Observed LTV': f'${l.observed_avg_ltv_usd:.2f}',
        'Projected LTV': f'${l.projected_ltv_usd:.2f}' if l.projected_ltv_usd else 'n/a',
        'LTV:CAC (Obs)': f'{l.ltv_cac_ratio_observed:.2f}x' if l.ltv_cac_ratio_observed else 'n/a',
        'LTV:CAC (Proj)': f'{l.ltv_cac_ratio_projected:.2f}x' if l.ltv_cac_ratio_projected else 'n/a',
        'Payback (mo)': f'{l.payback_months:.1f}' if l.payback_months else 'n/a',
        'Ranking Reversal': '⚠️ Yes' if l.ranking_reversal else ''
    })

df_ltv_cac = pd.DataFrame(ltv_cac_rows)
print('[SYNTHETIC DATA] LTV:CAC and Payback by Channel')
print('CPAO = spend / activated owners (campaign launched ≤14 days of trial)')
df_ltv_cac

In [ ]:
# Show ranking comparison explicitly
print('=== Ranking Comparison: Trial CAC vs CPAO vs Observed LTV ===')
print()

# Trial CAC from daily performance
con = duckdb.connect()
perf_path = str(DATA_DIR / 'fact_daily_performance.parquet')
df_trial_cac = con.execute(f"""
    SELECT channel,
           SUM(spend_usd) as total_spend,
           SUM(trial_signups) as total_trials,
           SUM(spend_usd) / NULLIF(SUM(trial_signups), 0) as trial_cac,
           ROW_NUMBER() OVER (ORDER BY SUM(spend_usd) / NULLIF(SUM(trial_signups), 0)) as trial_cac_rank
    FROM read_parquet('{perf_path}')
    GROUP BY channel
    ORDER BY trial_cac
""").df()
con.close()

print('Trial CAC Ranking (lower = better):')
for _, r in df_trial_cac.iterrows():
    print(f'  Rank {int(r.trial_cac_rank)}: {r.channel} — Trial CAC ${r.trial_cac:.2f}')

print()
print('CPAO Ranking (lower CPAO = more efficient):')
efficiency_channels = [(l.channel, l.cpao_usd) for l in ltv_by_ch if l.cpao_usd]
for rank, (ch, cpao) in enumerate(sorted(efficiency_channels, key=lambda x: x[1]), 1):
    print(f'  Rank {rank}: {ch} — CPAO ${cpao:.2f}')

print()
print('Observed LTV Ranking (higher LTV = better):')
for rank, l in enumerate(sorted(ltv_by_ch, key=lambda x: x.observed_avg_ltv_usd, reverse=True), 1):
    print(f'  Rank {rank}: {l.channel} — Observed LTV ${l.observed_avg_ltv_usd:.2f}')

**Ranking reversal — META:**
META and GOOGLE_SEARCH have similar trial CAC (~$343 and ~$354 respectively), but META's CPAO is $1,396 vs GOOGLE_SEARCH's $1,369 — and META shows the lowest observed LTV ($185.93) of the tracked channels. A media plan optimized purely on trial CAC would overweight META; adding activation and LTV data shifts the recommendation toward TIKTOK.

---
## 6. Budget Allocation Scenario ($150K/month, 10% Exploration)

**Allocation method:** Efficiency budget (90% = $135K) split proportional to `1/CPAO` — lower CPAO channels receive more budget. Max/min constraints prevent over-concentration. Exploration reserve (10% = $15K) goes to channels with no paid conversion history (LINKEDIN, UNKNOWN bucket).

**This is a recommendation only (`RECOMMEND_ONLY=True`). It should be validated with incrementality testing before committing.**

In [ ]:
scenario = AllocationScenario(
    scenario_name='Q4 2026 Baseline — CPAO Efficiency + 10% Exploration Reserve',
    total_budget_usd=150_000.0,
    exploration_reserve_pct=0.10,
    constraints=[
        ChannelConstraint('META', min_allocation_pct=0.10, max_allocation_pct=0.50),
        ChannelConstraint('GOOGLE_SEARCH', min_allocation_pct=0.10, max_allocation_pct=0.50),
        ChannelConstraint('TIKTOK', min_allocation_pct=0.05, max_allocation_pct=0.60),
        ChannelConstraint('LINKEDIN', min_allocation_pct=0.00, max_allocation_pct=0.20),
    ],
    optimize_by='cpao',
)

result = allocate_budget(scenario, DATA_DIR)

print(f'Scenario: {result.scenario_name}')
print(f'Total Budget: ${result.total_budget_usd:,.0f}')
print(f'Exploration Reserve: ${result.exploration_reserve_usd:,.0f} ({result.exploration_reserve_usd/result.total_budget_usd:.0%})')
print(f'Optimize By: {result.optimize_by.upper()}')
print(f'RECOMMEND_ONLY: {result.recommend_only}')
print()

alloc_rows = []
for a in result.allocations:
    alloc_rows.append({
        'Channel': a.channel,
        'CPAO': f'${a.cpao_usd:.0f}' if a.cpao_usd else 'n/a',
        'Recommended Spend': f'${a.recommended_spend_usd:,.0f}',
        'Pct of Budget': f'{a.recommended_pct:.1%}',
        'Expected Activated Owners': f'{a.expected_activated_owners:.0f}' if a.expected_activated_owners else 'n/a',
        'Type': 'Exploration' if a.is_exploration else 'Efficiency'
    })

df_alloc = pd.DataFrame(alloc_rows)
print('[SYNTHETIC DATA] Budget Allocation Recommendation')
df_alloc

In [ ]:
print('Allocation Notes:')
for note in result.notes:
    print(f'  • {note}')
print()

total_alloc = sum(a.recommended_spend_usd for a in result.allocations)
print(f'Budget check: ${total_alloc:,.2f} allocated vs ${result.total_budget_usd:,.0f} total')

---
## 7. Key Findings: What Changes When You Add Retention to the Acquisition Picture?

This section synthesizes the Phase 6 analysis into three decision-relevant observations.

In [ ]:
print("""
=============================================================
KEY FINDINGS — Phase 6 [SYNTHETIC DATA]
=============================================================

1. TIKTOK IS MORE EFFICIENT THAN IT APPEARS BY TRIAL METRICS
   ----------------------------------------------------------
   By trial CAC, META and GOOGLE_SEARCH look comparable to TIKTOK.
   But TIKTOK's CPAO is $908 vs $1,369 (GOOGLE_SEARCH) and $1,396 (META).
   This means TIKTOK delivers activated owners at 54% lower cost than META.
   Adding retention further widens the gap: TIKTOK M6 retention is 47.4%
   vs META's 43.2% and GOOGLE_SEARCH's 42.9%.

2. META SHOWS A DOWNSTREAM RANKING REVERSAL
   -----------------------------------------
   META ranks 2nd by trial CAC efficiency but last by observed LTV ($185.93)
   and near-last by M6 retention (43.2%). This is the classic ranking reversal:
   a channel that converts clicks to trials efficiently but fails to convert
   trials to long-term value. Budget decisions made on trial CAC alone would
   overallocate to META.

3. RIGHT CENSORING MAKES M6 PROVISIONAL
   --------------------------------------
   All M6 rates carry an immature warning because April–May 2026 conversions
   have not been observable for 180 days as of the reference date (2026-09-13).
   M6 rates are computed on 78–133 eligible subscribers per channel, not the
   full 150–227 subscriber base. Final M6 rates should be re-evaluated when
   all cohorts mature (approximately October–November 2026).

4. OBSERVED LTV IS UNDERSTATED — USE WITH CAUTION
   ------------------------------------------------
   The 2–3 month revenue observation window means observed LTV ($186–$212)
   captures only the initial MRR events. Projected LTV (constant-churn model)
   is $306–$374, but this assumes a fixed churn rate that does not reflect
   real early-period churn being higher than steady-state.

5. EXPLORATION BUDGET IS NON-NEGOTIABLE
   --------------------------------------
   LINKEDIN has no conversion data in the synthetic dataset. Allocating 0%
   would eliminate a potentially valuable channel permanently. The 10%
   exploration reserve ($15K/month) preserves optionality and enables
   future CPAO measurement on untested channels.
""")

---
**[SYNTHETIC DATA — Tablr portfolio simulation]**

All data was generated programmatically to demonstrate analytical methodology. No real Tablr business data was used.

**Maturity reference date:** 2026-09-13 | **Simulation period:** 2026-01-05 to 2026-03-29